# Chapitre 9 — DocuRAG

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahouahounko/rag-en-pratique/blob/main/chapters/chapitre-09-docurag/09_docurag.ipynb)

DocuRAG organise le pipeline du chapitre 2 en application modulaire utilisant OpenAI pour les embeddings et la génération.

## Scripts et fichiers du chapitre

1. [`01_arborescence.txt`](examples/01_arborescence.txt)
2. [`02_config.py`](examples/02_config.py)
3. [`03_env_example.txt`](examples/03_env_example.txt)
4. [`04_requirements.txt`](examples/04_requirements.txt)
5. [`05_loader.py`](examples/05_loader.py)
6. [`06_chunker.py`](examples/06_chunker.py)
7. [`07_indexer.py`](examples/07_indexer.py)
8. [`08_pipeline_ingestion.py`](examples/08_pipeline_ingestion.py)
9. [`09_retriever.py`](examples/09_retriever.py)
10. [`10_prompts.py`](examples/10_prompts.py)
11. [`11_generator.py`](examples/11_generator.py)
12. [`12_schemas.py`](examples/12_schemas.py)
13. [`13_api_main.py`](examples/13_api_main.py)
14. [`14_interface.py`](examples/14_interface.py)
15. [`15_test_evaluation.py`](examples/15_test_evaluation.py)
16. [`16_dockerfile.txt`](examples/16_dockerfile.txt)
17. [`17_compose.yml`](examples/17_compose.yml)
18. [`18_demarrage.sh`](examples/18_demarrage.sh)

## 1. Préparer le dépôt

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

if not Path("src").is_dir():
    if not Path("rag-en-pratique").is_dir():
        subprocess.run(["git", "clone", "https://github.com/Ahouahounko/rag-en-pratique.git"], check=True)
    os.chdir("rag-en-pratique")

sys.path.insert(0, str(Path("src").resolve()))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("Dépôt prêt :", Path.cwd())


## 2. Configurer OpenAI

In [ ]:
from getpass import getpass
import os

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY : ")
if not os.getenv("OPENAI_MODEL"):
    os.environ["OPENAI_MODEL"] = input("OPENAI_MODEL : ").strip()

if not os.environ["OPENAI_API_KEY"] or not os.environ["OPENAI_MODEL"]:
    raise RuntimeError("OPENAI_API_KEY et OPENAI_MODEL sont obligatoires")
print("Configuration OpenAI chargée.")


## 3. Charger l'application DocuRAG

Configuration, chargement et chunking correspondent aux exemples 02 à 06.

In [ ]:
from pathlib import Path
import sys

runnable = Path("chapters/chapitre-09-docurag/runnable").resolve()
sys.path.insert(0, str(runnable))

from docurag import DocuRAG
from docurag.config import Settings

settings = Settings.from_env()
app = DocuRAG(settings)


## 4. Ingérer et indexer un dossier

Correspond aux exemples 07 et 08. Les embeddings sont calculés par OpenAI.

In [ ]:
chunk_count = app.ingest(Path("data/sample"))
print(f"{chunk_count} chunks indexés")


## 5. Interroger DocuRAG

Correspond aux exemples 09 à 13 : retrieval, prompts, génération, schémas et API.

In [ ]:
result = app.ask("Quel est le délai de livraison standard ?")
print(result["answer"])


## 6. Inspecter la traçabilité

In [ ]:
for rank, source in enumerate(result["sources"], start=1):
    print(f"#{rank} score={source['score']} source={source['metadata']['source']}")
    print(source["text"][:300])
    print()


## 7. Tester une question absente des documents

In [ ]:
unknown = app.ask("Quel est le numéro de téléphone du directeur ?")
print(unknown["answer"])


## 8. Interface, évaluation et déploiement

Les exemples 14 à 18 couvrent Streamlit, l'évaluation, Docker, Compose et le démarrage.

## Architecture

- `config.py` : configuration explicite ;
- `loaders.py` : chargement Markdown, texte et PDF facultatif ;
- `pipeline.py` : ingestion, retrieval et génération ;
- `cli.py` : interface en ligne de commande ;
- `rag_en_pratique.core` : composants partagés et testables.